In [5]:
# describe_tree_notebook.py
# À exécuter tel quel dans un notebook Jupyter

from pathlib import Path

# 🔧 CHEMIN À EXPLORER (modifiable directement ici)
ROOT_PATH = Path(r"C:\Users\Sid\Desktop\Le Risque Tout\V9\Quizz\questions\disney")

def print_tree(path: Path, prefix: str = ""):
    entries = sorted(
        list(path.iterdir()),
        key=lambda x: (x.is_file(), x.name.lower())
    )

    for idx, entry in enumerate(entries):
        is_last = idx == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + entry.name)

        if entry.is_dir():
            extension = "    " if is_last else "│   "
            print_tree(entry, prefix + extension)

def describe_project_tree(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"Le chemin n'existe pas : {root}")

    if not root.is_dir():
        raise NotADirectoryError(f"Le chemin n'est pas un dossier : {root}")

    print(root)
    print_tree(root)


# ▶️ Exécution
describe_project_tree(ROOT_PATH)


C:\Users\Sid\Desktop\Le Risque Tout\V9\Quizz\questions\disney
├── 1
│   └── images
│       ├── Anna.webp
│       ├── Ariel.webp
│       ├── Bambi.webp
│       ├── Belle.webp
│       ├── Black_Panther.jpeg
│       ├── Black_Widow.webp
│       ├── Buzz_Lightyear.webp
│       ├── Captain_America.jpg
│       ├── Doctor_Strange.webp
│       ├── Donald_Duck.webp
│       ├── Dory.webp
│       ├── Elsa.webp
│       ├── Goofy.webp
│       ├── Hulk.webp
│       ├── Iron_Man.webp
│       ├── Joy.webp
│       ├── Lightning_McQueen.webp
│       ├── Loki.webp
│       ├── Mickey_Mouse.webp
│       ├── Minnie_Mouse.webp
│       ├── Nemo.webp
│       ├── Pluto.webp
│       ├── Pumbaa.webp
│       ├── Simba.webp
│       ├── Spider-Man.webp
│       ├── Thanos.webp
│       ├── Thor.webp
│       ├── Timon.webp
│       └── Woody.webp
├── 2
│   └── images
│       ├── Aladdin.webp
│       ├── Ant-Man.webp
│       ├── Captain_Marvel.webp
│       ├── Carl_Fredricksen.webp
│       ├── Gamora.webp
│       ├── Gen

In [6]:
from __future__ import annotations

from pathlib import Path

ROOT = Path(r"C:\Users\Sid\Desktop\Le Risque Tout\V9\Quizz\questions\disney")
ALLOWED_EXTS = {".webp", ".jpg", ".jpeg", ".png"}

MAP_EN_TO_FR = {
    # 1
    "Buzz_Lightyear": "Buzz_LEclair",
    "Goofy": "Dingo",
    "Joy": "Joie",
    "Lightning_McQueen": "Flash_McQueen",

    # 2
    "Hawkeye": "Oeil_de_Faucon",
    "Rapunzel": "Raiponce",
    "Sadness": "Tristesse",
    "Snow_White": "Blanche_Neige",
    "Tinker_Bell": "Clochette",
    "Violet_Parr": "Violette_Parr",
    "Wasp": "La_Guepe",
    "Winnie_the_Pooh": "Winnie_l_Ourson",

    # 3
    "Anger": "Colere",
    "Captain_Hook": "Capitaine_Crochet",
    "Disgust": "Degout",
    "Falcon": "Faucon",
    "Fear": "Peur",
    "Flik": "Tilt",
    "Maleficent": "Malefique",
    "Rocket_Raccoon": "Rocket_Raton_Laveur",
    "Tigger": "Tigrou",
    "Winter_Soldier": "Soldat_de_l_Hiver",

    # 4
    "Aurora": "Aurore",
    "Dug": "Doug",
    "Moana": "Vaiana",
    "Red_Skull": "Crane_Rouge",

    # 5
    "Doctor_Octopus": "Docteur_Octopus",
    "Green_Goblin": "Bouffon_Vert",
    "Queen_of_Hearts": "Reine_de_Coeur",
    "Scrooge_McDuck": "Balthazar_Picsou",
    "Thumper": "Panpan",
}

def build_plan(root: Path) -> list[tuple[Path, Path]]:
    plan: list[tuple[Path, Path]] = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in ALLOWED_EXTS:
            continue

        stem = p.stem
        new_stem = MAP_EN_TO_FR.get(stem)
        if not new_stem or new_stem == stem:
            continue

        plan.append((p, p.with_name(new_stem + p.suffix)))
    return plan


def validate_plan(plan: list[tuple[Path, Path]]) -> None:
    # collisions: plusieurs sources -> même cible
    targets: dict[str, list[tuple[Path, Path]]] = {}
    for src, dst in plan:
        targets.setdefault(str(dst).lower(), []).append((src, dst))

    collisions = {k: v for k, v in targets.items() if len(v) > 1}
    if collisions:
        print("ERREUR: collisions détectées :")
        for _, pairs in collisions.items():
            print(f"  Cible: {pairs[0][1]}")
            for src, _ in pairs:
                print(f"    - {src}")
        raise SystemExit(1)

    # conflit: la cible existe déjà
    existing = [(src, dst) for src, dst in plan if dst.exists()]
    if existing:
        print("ERREUR: certaines cibles existent déjà :")
        for src, dst in existing:
            print(f"  {src} -> {dst} (déjà présent)")
        raise SystemExit(1)


def apply(plan: list[tuple[Path, Path]], dry_run: bool = True) -> None:
    if not plan:
        print("Aucun fichier à renommer.")
        return

    validate_plan(plan)

    if dry_run:
        print("DRY RUN (aucun changement). Renommages prévus :")
        for src, dst in plan:
            print(f"  {src} -> {dst}")
        return

    for src, dst in plan:
        src.rename(dst)
        print(f"OK: {src.name} -> {dst.name}")


if __name__ == "__main__":
    DRY_RUN = False  # mets True si tu veux re-vérifier
    apply(build_plan(ROOT), dry_run=DRY_RUN)


OK: Buzz_Lightyear.webp -> Buzz_LEclair.webp
OK: Goofy.webp -> Dingo.webp
OK: Joy.webp -> Joie.webp
OK: Lightning_McQueen.webp -> Flash_McQueen.webp
OK: Hawkeye.webp -> Oeil_de_Faucon.webp
OK: Rapunzel.webp -> Raiponce.webp
OK: Sadness.jpg -> Tristesse.jpg
OK: Snow_White.webp -> Blanche_Neige.webp
OK: Tinker_Bell.webp -> Clochette.webp
OK: Violet_Parr.webp -> Violette_Parr.webp
OK: Wasp.webp -> La_Guepe.webp
OK: Winnie_the_Pooh.webp -> Winnie_l_Ourson.webp
OK: Anger.jpg -> Colere.jpg
OK: Captain_Hook.webp -> Capitaine_Crochet.webp
OK: Disgust.jpg -> Degout.jpg
OK: Falcon.webp -> Faucon.webp
OK: Fear.jpg -> Peur.jpg
OK: Flik.webp -> Tilt.webp
OK: Maleficent.webp -> Malefique.webp
OK: Rocket_Raccoon.webp -> Rocket_Raton_Laveur.webp
OK: Tigger.webp -> Tigrou.webp
OK: Winter_Soldier.webp -> Soldat_de_l_Hiver.webp
OK: Aurora.webp -> Aurore.webp
OK: Dug.webp -> Doug.webp
OK: Moana.webp -> Vaiana.webp
OK: Red_Skull.webp -> Crane_Rouge.webp
OK: Doctor_Octopus.webp -> Docteur_Octopus.webp
OK: G